## 1. Setup Colab

**Runtime > Modifier le type d'exécution > GPU** avant de commencer.

Remplacez `OWNER` par votre nom d'utilisateur GitHub.

In [ ]:
# Cloner le dépôt puis s'y placer.
!git clone https://github.com/OWNER/cross-domain-visual-anomaly-detection.git
%cd cross-domain-visual-anomaly-detection

In [ ]:
# Installer le package (PyTorch est déjà présent sur Colab).
!pip -q install -e .

In [ ]:
# Vérifier le GPU.
import torch

print(
    "CUDA:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
)

In [ ]:
# (Optionnel) Monter Google Drive pour sauvegarder checkpoints et métriques.
# from google.colab import drive
# drive.mount('/content/drive')


## 2. Données

Comme pour l'autoencodeur : synthétique pour l'essai, MVTec pour le réel.

In [ ]:
!python scripts/download_data.py --synthetic
CATEGORY = "synthetic"
# CATEGORY = "bottle"  # après téléchargement de MVTec

## 3. Construction de la banque mémoire PatchCore

PatchCore n'entraîne rien par gradient : il extrait des features d'un
backbone pré-entraîné (téléchargé au premier appel) et construit une
banque mémoire, sous-échantillonnée par coreset.

In [ ]:
!python scripts/train.py --model patchcore --category $CATEGORY --profile colab_gpu

## 4. Évaluation et comparaison avec l'autoencodeur

Entraînez d'abord l'autoencodeur (notebook 01) pour disposer des deux
checkpoints, puis comparez sous le même protocole.

In [ ]:
!python scripts/evaluate.py --category $CATEGORY \
  --checkpoint models/industrial_patchcore_$CATEGORY.pt

In [ ]:
# Comparaison autoencodeur vs PatchCore (nécessite les deux checkpoints).
!python scripts/compare_models.py --category $CATEGORY \
  --ae-checkpoint models/industrial_autoencoder_$CATEGORY.pt \
  --patchcore-checkpoint models/industrial_patchcore_$CATEGORY.pt

## Récupérer les artefacts en local

Les modèles/métriques sont dans `models/` et `reports/`. Copiez-les vers
Drive (ci-dessus) ou téléchargez-les, puis committez-les **depuis votre
machine** (ne poussez pas de gros checkpoints ; préférez Drive).

In [ ]:
# Copier les artefacts vers Drive (si monté).
# !mkdir -p /content/drive/MyDrive/anomaly_artifacts
# !cp -r models reports /content/drive/MyDrive/anomaly_artifacts/


In [ ]:
# Commande git à exécuter EN LOCAL après récupération des artefacts :
print(r"""git add reports/ docs/ && \
git commit -m "docs(report): add real experiment metrics from Colab" && \
git push origin main""")

## Reprise après interruption

Colab peut se déconnecter. Les checkpoints sont sauvegardés dans `models/`.
Après reconnexion, ré-exécutez le Setup puis relancez la cellule
d'entraînement : un checkpoint existant sert de point de reprise / base
d'évaluation (l'entraînement complet reste à relancer si non terminé).